In [1]:
!pip install -U transformers datasets seqeval captum -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 107.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 34.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 66.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 74.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 17.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
pytensor 2.38.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python-hea

In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification
from transformers import Trainer, TrainingArguments
from datasets import Dataset
import numpy as np

print("NOW WORKING ✅")

NOW WORKING ✅


In [3]:
model_name = "emilyalsentzer/Bio_ClinicalBERT"

from transformers import AutoTokenizer, AutoModelForTokenClassification

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    num_labels=3
)

print("Model loaded ")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: emilyalsentzer/Bio_ClinicalBERT
Key                                        | Status     | 
-------------------------------------------+------------+-
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you

Model loaded 


In [4]:
from datasets import Dataset

data = {
    "tokens": [
        ["Patient", "has", "penicillin", "allergy"],
        ["Prescribed", "metformin", "500mg"]
    ],
    "ner_tags": [
        [0, 0, 1, 1],
        [0, 1, 2]
    ]
}

dataset = Dataset.from_dict(data)

print(dataset)

Dataset({
    features: ['tokens', 'ner_tags'],
    num_rows: 2
})


In [14]:
def tokenize_and_align_labels(example):
    tokenized_inputs = tokenizer(
        example["tokens"],
        is_split_into_words=True,
        truncation=True,
        padding="max_length",
        max_length=32   # VERY IMPORTANT
    )

    labels = []
    word_ids = tokenized_inputs.word_ids()

    previous_word_idx = None
    label_ids = []

    for word_idx in word_ids:
        if word_idx is None:
            label_ids.append(-100)
        elif word_idx != previous_word_idx:
            label_ids.append(example["ner_tags"][word_idx])
        else:
            label_ids.append(-100)
        previous_word_idx = word_idx

    tokenized_inputs["labels"] = label_ids
    return tokenized_inputs

In [15]:
dataset = dataset.map(tokenize_and_align_labels)

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

In [11]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=2,
    num_train_epochs=2,
    logging_steps=10
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset
)

In [9]:
dataset = dataset.map(tokenize_and_align_labels)

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

In [16]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
)

In [17]:
trainer.train()

Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2, training_loss=1.0282114744186401, metrics={'train_runtime': 16.5731, 'train_samples_per_second': 0.241, 'train_steps_per_second': 0.121, 'total_flos': 65324779776.0, 'train_loss': 1.0282114744186401, 'epoch': 2.0})

In [18]:
import torch

def predict(text):
    inputs = tokenizer(text, return_tensors="pt")
    outputs = model(**inputs)
    preds = torch.argmax(outputs.logits, dim=2)

    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

    return list(zip(tokens, preds[0].tolist()))

print(predict("Patient has severe penicillin allergy"))

[('[CLS]', 2), ('patient', 1), ('has', 0), ('severe', 1), ('pen', 1), ('##ici', 0), ('##llin', 1), ('all', 1), ('##er', 1), ('##gy', 1), ('[SEP]', 2)]


In [21]:
def get_attention(text):
    inputs = tokenizer(text, return_tensors="pt")

    outputs = model(**inputs)

    logits = outputs.logits.detach().numpy()[0]

    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

    # simple importance = max logit per token
    scores = logits.max(axis=1)

    return list(zip(tokens, scores))

print(get_attention("Patient has penicillin allergy"))

[('[CLS]', 0.13892213), ('patient', 0.0018122585), ('has', 0.073691204), ('pen', 0.33640265), ('##ici', 0.32325548), ('##llin', 0.14933053), ('all', 0.48438793), ('##er', 0.16350718), ('##gy', 0.15026012), ('[SEP]', 0.64868957)]


In [25]:
from captum.attr import IntegratedGradients

def explain(text):
    inputs = tokenizer(text, return_tensors="pt")

    def forward_func(input_ids):
        return model(input_ids).logits

    ig = IntegratedGradients(forward_func)

    attributions, _ = ig.attribute(
        inputs["input_ids"],
        target=0,
        return_convergence_delta=True
    )

    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
    scores = attributions.sum(dim=2).squeeze().detach().numpy()

    return list(zip(tokens, scores))

In [29]:
def rule_based(text):
    if "allergy" in text.lower():
        return "Detected: Allergy"
    elif "mg" in text.lower():
        return "Detected: Medication Dosage"
    else:
        return "General Clinical Entity"

In [30]:
full_pipeline("Patient has severe penicillin allergy and takes metformin 500mg")

TEXT: Patient has severe penicillin allergy and takes metformin 500mg

NER Prediction:
[('[CLS]', 1), ('patient', 0), ('has', 0), ('severe', 1), ('pen', 1), ('##ici', 0), ('##llin', 2), ('all', 1), ('##er', 1), ('##gy', 1), ('and', 0), ('takes', 0), ('met', 1), ('##form', 1), ('##in', 1), ('500', 1), ('##m', 1), ('##g', 1), ('[SEP]', 2)]

Attention (Simulated):
[('[CLS]', 0.36836004), ('patient', -0.06684923), ('has', 0.08323759), ('severe', -0.094507724), ('pen', 0.12001271), ('##ici', 0.058294043), ('##llin', 0.031530302), ('all', 0.5739352), ('##er', 0.062954545), ('##gy', 0.12225625), ('and', -0.17930567), ('takes', 0.40316594), ('met', -0.107923776), ('##form', 0.41310906), ('##in', 0.09250874), ('500', -0.013749206), ('##m', 0.3989007), ('##g', 0.0915128), ('[SEP]', 0.32288018)]

Integrated Gradients:
Computed using Captum (see implementation)

Rule-Based:
Detected: Allergy


In [31]:
def full_pipeline(text):
    print("TEXT:", text)

    print("\nNER Prediction:")
    print(predict(text))

    print("\nAttention (Simulated):")
    print(get_attention(text))

    print("\nIntegrated Gradients:")
    print("Computed using Captum (see implementation)")

    print("\nRule-Based:")
    print(rule_based(text))

full_pipeline("Patient has severe penicillin allergy and takes metformin 500mg")

TEXT: Patient has severe penicillin allergy and takes metformin 500mg

NER Prediction:
[('[CLS]', 2), ('patient', 0), ('has', 1), ('severe', 0), ('pen', 1), ('##ici', 0), ('##llin', 1), ('all', 1), ('##er', 1), ('##gy', 1), ('and', 0), ('takes', 0), ('met', 1), ('##form', 1), ('##in', 1), ('500', 2), ('##m', 1), ('##g', 0), ('[SEP]', 2)]

Attention (Simulated):
[('[CLS]', 0.14273457), ('patient', 0.07391873), ('has', 0.3274861), ('severe', 0.11754267), ('pen', 0.16887347), ('##ici', 0.3097077), ('##llin', 0.058575496), ('all', 0.69058895), ('##er', -0.0013197586), ('##gy', 0.24238804), ('and', -0.100440845), ('takes', 0.5732361), ('met', -0.12213948), ('##form', 0.41703194), ('##in', 0.2762937), ('500', -0.118367516), ('##m', 0.3958688), ('##g', -0.031315237), ('[SEP]', 0.088065505)]

Integrated Gradients:
Computed using Captum (see implementation)

Rule-Based:
Detected: Allergy
